# 01 — Allocate the population aged 65+ to residential buildings

This notebook reproduces the demographic allocation used in the manuscript.

**Purpose**

1. Load 2021 BGRI statistical subsections and residential building footprints.
2. Assign each building to one statistical subsection.
3. Allocate the population aged 65 years and over in proportion to building footprint area.
4. Convert proportional estimates to integer counts and reconcile rounding differences within each subsection.
5. Export the building-level population file used by the downstream accessibility analysis.

The output is an **operational estimate**, not an observed building-level population count. Building footprint area is used as a proxy for residential capacity because consistent information on building height, number of storeys and residential typology is not available for the full study area.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely import wkt
CRS_GEOGRAPHIC = 'EPSG:4326'
CRS_PROJECTED = 'EPSG:3763'

def resolve_repo_root():
    cwd = Path.cwd().resolve()
    if cwd.name == 'notebooks':
        return cwd.parent
    if (cwd / 'notebooks').exists() and (cwd / 'data').exists():
        return cwd
    for parent in cwd.parents:
        if (parent / 'notebooks').exists() and (parent / 'data').exists():
            return parent
    raise RuntimeError('Repository root not found. Run the notebook from the repository root or from the notebooks directory.')
REPO_ROOT = resolve_repo_root()
RAW_DATA = REPO_ROOT / 'data' / 'raw'
INTERMEDIATE_DATA = REPO_ROOT / 'data' / 'intermediate' / 'population'
INTERMEDIATE_DATA.mkdir(parents=True, exist_ok=True)
BGRI_FILE = RAW_DATA / 'BGRI2021_1312.gpkg'
BUILDINGS_FILE = RAW_DATA / 'edificios.csv'
OUTPUT_FILE = INTERMEDIATE_DATA / 'population_65plus_by_building.csv'
AUDIT_FILE = INTERMEDIATE_DATA / 'population_65plus_allocation_audit.csv'
BGRI_POP_COL = 'N_INDIVIDUOS_65_OU_MAIS'
BGRI_ID_CANDIDATES = ['DTMNFRSEC21', 'SUBSECCAO']
STRICT_MANUSCRIPT_VALIDATION = True
EXPECTED_MUNICIPAL_65PLUS = 60216
EXPECTED_ALLOCATED_TO_ELIGIBLE_BUILDINGS = 58748
for path in [BGRI_FILE, BUILDINGS_FILE]:
    if not path.exists():
        raise FileNotFoundError(f'Missing required input: {path}')
print('Repository root:', REPO_ROOT)
print('BGRI:', BGRI_FILE)
print('Buildings:', BUILDINGS_FILE)
print('Output:', OUTPUT_FILE)

## 1. Load and validate the spatial inputs

Building geometries are reconstructed from WKT and projected to EPSG:3763. Invalid geometries are repaired with a zero-width buffer. Each building must receive a single BGRI subsection assignment.


In [ ]:
bgri = gpd.read_file(BGRI_FILE).to_crs(CRS_PROJECTED)
if BGRI_POP_COL not in bgri.columns:
    raise KeyError(f'{BGRI_POP_COL} is not present in the BGRI file. Available columns: {bgri.columns.tolist()}')
bgri_id_col = next((c for c in BGRI_ID_CANDIDATES if c in bgri.columns), None)
if bgri_id_col is None:
    raise KeyError(f'No recognised BGRI subsection identifier was found. Tried: {BGRI_ID_CANDIDATES}')
buildings_df = pd.read_csv(BUILDINGS_FILE, low_memory=False)
geometry_col = next((c for c in ['geometry_wkt', 'geometry'] if c in buildings_df.columns), None)
if geometry_col is None:
    raise KeyError('The building file must contain geometry_wkt or geometry.')
buildings_df['geometry'] = buildings_df[geometry_col].apply(lambda value: wkt.loads(str(value)) if pd.notna(value) and str(value).strip() else None)
buildings = gpd.GeoDataFrame(buildings_df, geometry='geometry', crs=CRS_GEOGRAPHIC).dropna(subset=['geometry']).to_crs(CRS_PROJECTED)
buildings['geometry'] = buildings.geometry.buffer(0)
bgri['geometry'] = bgri.geometry.buffer(0)
buildings['osm_id'] = buildings['osm_id'].astype(str).str.replace('\\.0$', '', regex=True).str.strip()
if buildings['osm_id'].duplicated().any():
    raise ValueError('Duplicate osm_id values are present before BGRI assignment.')
municipal_total = int(pd.to_numeric(bgri[BGRI_POP_COL], errors='coerce').fillna(0).sum())
print('Residential building records:', len(buildings))
print('BGRI subsections:', len(bgri))
print('Municipal population aged 65+:', municipal_total)
if STRICT_MANUSCRIPT_VALIDATION:
    assert municipal_total == EXPECTED_MUNICIPAL_65PLUS, f'Expected {EXPECTED_MUNICIPAL_65PLUS} residents aged 65+, found {municipal_total}.'

## 2. Assign buildings to BGRI subsections

The original workflow used polygon intersection. To keep a unique building universe, the code first performs that same spatial intersection and resolves the rare case of a building intersecting more than one subsection by retaining the subsection with the largest building–subsection overlap area.


In [ ]:
bgri_small = bgri[[bgri_id_col, BGRI_POP_COL, 'geometry']].copy()
joined = gpd.sjoin(buildings[['osm_id', 'geometry']].copy(), bgri_small[[bgri_id_col, 'geometry']], how='left', predicate='intersects')
duplicate_ids = joined.loc[joined['osm_id'].duplicated(keep=False), 'osm_id'].unique()
if len(duplicate_ids):
    candidates = joined[joined['osm_id'].isin(duplicate_ids)].copy()
    candidates = candidates.drop(columns='index_right', errors='ignore')
    candidates = candidates.merge(bgri_small[[bgri_id_col, 'geometry']].rename(columns={'geometry': '_bgri_geometry'}), on=bgri_id_col, how='left')
    candidates['_overlap_area'] = candidates.apply(lambda row: row['geometry'].intersection(row['_bgri_geometry']).area if row['_bgri_geometry'] is not None else -1, axis=1)
    chosen = candidates.sort_values(['osm_id', '_overlap_area', bgri_id_col], ascending=[True, False, True]).drop_duplicates('osm_id')[['osm_id', bgri_id_col]]
    single = joined[~joined['osm_id'].isin(duplicate_ids)][['osm_id', bgri_id_col]]
    assignment = pd.concat([single, chosen], ignore_index=True)
else:
    assignment = joined[['osm_id', bgri_id_col]].copy()
assignment = assignment.drop_duplicates('osm_id')
if assignment['osm_id'].duplicated().any():
    raise RuntimeError('BGRI assignment is not unique after duplicate resolution.')
buildings = buildings.merge(assignment, on='osm_id', how='left', validate='one_to_one')
n_unassigned = int(buildings[bgri_id_col].isna().sum())
print('Buildings assigned to a BGRI subsection:', len(buildings) - n_unassigned)
print('Buildings without BGRI assignment:', n_unassigned)
print('Buildings intersecting >1 subsection before resolution:', len(duplicate_ids))

## 3. Proportional areal interpolation and integer reconciliation

Within each subsection, population is distributed according to each building's share of the total residential footprint area. Proportional estimates are rounded to the nearest integer. Any resulting difference from the official subsection total is reconciled deterministically using the rounding residuals while preventing negative building-level counts.


In [ ]:
allocation = buildings.dropna(subset=[bgri_id_col]).copy()
allocation['footprint_area_m2'] = allocation.geometry.area
population_by_subsection = bgri_small[[bgri_id_col, BGRI_POP_COL]].drop_duplicates(bgri_id_col).copy()
population_by_subsection[BGRI_POP_COL] = pd.to_numeric(population_by_subsection[BGRI_POP_COL], errors='coerce').fillna(0).astype(int)
allocation = allocation.merge(population_by_subsection, on=bgri_id_col, how='left', validate='many_to_one')
allocation['subsection_building_area_m2'] = allocation.groupby(bgri_id_col)['footprint_area_m2'].transform('sum')
allocation['population_65plus_raw'] = np.where(allocation['subsection_building_area_m2'] > 0, allocation['footprint_area_m2'] / allocation['subsection_building_area_m2'] * allocation[BGRI_POP_COL], 0.0)
allocation['pop_64_mais'] = np.rint(allocation['population_65plus_raw']).astype(int)

def reconcile_subsection(group):
    group = group.copy()
    official = int(group[BGRI_POP_COL].iloc[0])
    current = int(group['pop_64_mais'].sum())
    difference = official - current
    if difference == 0:
        return group
    group['_rounding_residual'] = group['population_65plus_raw'] - group['pop_64_mais']
    if difference > 0:
        order = group.sort_values(['_rounding_residual', 'population_65plus_raw', 'osm_id'], ascending=[False, False, True]).index.tolist()
        for i in range(difference):
            group.loc[order[i % len(order)], 'pop_64_mais'] += 1
    else:
        remaining = abs(difference)
        candidates = group[group['pop_64_mais'] > 0].sort_values(['_rounding_residual', 'population_65plus_raw', 'osm_id'], ascending=[True, True, True]).index.tolist()
        if not candidates and remaining:
            raise RuntimeError(f'Cannot reconcile subsection {group[bgri_id_col].iloc[0]} without creating negative population values.')
        cursor = 0
        while remaining > 0:
            idx = candidates[cursor % len(candidates)]
            if group.loc[idx, 'pop_64_mais'] > 0:
                group.loc[idx, 'pop_64_mais'] -= 1
                remaining -= 1
            cursor += 1
    group['pop_64_mais'] = group['pop_64_mais'].clip(lower=0).astype(int)
    group = group.drop(columns='_rounding_residual', errors='ignore')
    return group
allocation = allocation.groupby(bgri_id_col, group_keys=False).apply(reconcile_subsection).reset_index(drop=True)
check = allocation.groupby(bgri_id_col, as_index=False).agg(official_65plus=(BGRI_POP_COL, 'first'), allocated_65plus=('pop_64_mais', 'sum'), n_buildings=('osm_id', 'count'), building_area_m2=('footprint_area_m2', 'sum'))
check['difference'] = check['allocated_65plus'] - check['official_65plus']
if not check['difference'].eq(0).all():
    raise AssertionError('At least one BGRI subsection was not reconciled exactly.')
allocated_total = int(allocation['pop_64_mais'].sum())
print('Population allocated to eligible residential buildings:', allocated_total)
print('Subsections represented by eligible buildings:', len(check))
print('Maximum absolute subsection reconciliation error:', int(check['difference'].abs().max()))
if STRICT_MANUSCRIPT_VALIDATION:
    assert allocated_total == EXPECTED_ALLOCATED_TO_ELIGIBLE_BUILDINGS, f'Expected {EXPECTED_ALLOCATED_TO_ELIGIBLE_BUILDINGS} allocated residents, found {allocated_total}.'

## 4. Export and final checks

The downstream analytical universe is defined after pedestrian-network validation. Therefore, this notebook exports all eligible building allocations; the final retained population total is checked after merging with the 31,664-building accessibility universe.


In [ ]:
output = allocation[['osm_id', bgri_id_col, 'footprint_area_m2', 'pop_64_mais']].copy()
output.to_csv(OUTPUT_FILE, index=False)
check.to_csv(AUDIT_FILE, index=False)
assert output['osm_id'].is_unique
assert (output['pop_64_mais'] >= 0).all()
print('Saved:', OUTPUT_FILE.resolve())
print('Audit:', AUDIT_FILE.resolve())
print('Rows:', len(output))
print('Allocated population aged 65+:', int(output['pop_64_mais'].sum()))
print('Population allocation validation: OK')